# Mechanistic Detection of LLM Perturbations

This notebook presents the four main experimental results:

1. **Localization (Main)** — Can models detect *which* sentence was perturbed
   (dropout or noise), as a function of perturbation strength?
2. **Localization (Control)** — Same setup but with topic-based prompts
   ("which sentence is about animals?") instead of perturbation detection.
3. **Zero-shot Classification** — Can models name the perturbation type
   (dropout vs noise) without any examples? Includes control aliases.
4. **ICL Learning Dynamics** — How does few-shot (in-context learning)
   accuracy scale with the number of teaching examples?

All data is loaded from the local wandb cache (see `CACHE_DIR` below).

---

## Behind the curtain

Install dependencies, load cached experiment data, and define plotting helpers.
Run these cells once, then jump to the experiment sections below.

In [1]:
%pip install -q ipywidgets matplotlib numpy pandas pyarrow

# ── Configuration ────────────────────────────────────────────
# Path to the folder containing cached sweep pickle files.
CACHE_DIR = "../data/cache"
# ─────────────────────────────────────────────────────────────

import os
import pathlib

CACHE_DIR = pathlib.Path(CACHE_DIR)

_this_dir = pathlib.Path(os.path.abspath("")).resolve()
if _this_dir.name == "paper":
    os.chdir(_this_dir.parent)

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D

# ── Cache loader ─────────────────────────────────────────────


def load_sweep(sweep_id, project=None):
    cache_path = CACHE_DIR / f"{sweep_id}.pkl"
    return pd.read_pickle(cache_path)


# ── Shared constants ─────────────────────────────────────────

PROJECT = "llm-mechanistic-detection"
MODELS = ["llama3_8b", "qwen3_14b", "qwen3_32b", "olmo3_32b"]

MODEL_LABELS = {
    "llama3_8b": "Llama 3 8B",
    "qwen3_14b": "Qwen 3 14B",
    "qwen3_32b": "Qwen 3 32B",
    "olmo3_32b": "OLMo 3 32B",
}

MODEL_COLORS = {
    "llama3_8b": "#f4b6c2",
    "qwen3_14b": "#8fae82",
    "qwen3_32b": "#1b3a6b",
    "olmo3_32b": "#8b1a1a",
}

COL_MODEL = "model"
COL_DROPOUT = "perturbation.dropout_rate"
COL_NOISE = "perturbation.noise_std"

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 11,
    }
)

/Users/m.bronzi/workspace/llm_mechanistic_detection/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ══════════════════════════════════════════════════════════════
# Data loading (all experiments)
# ══════════════════════════════════════════════════════════════


def load_and_merge(sweep_ids, project):
    dfs = [load_sweep(sid, project=project) for sid in sweep_ids]
    return pd.concat(dfs, ignore_index=True)


def _stem(path):
    return pathlib.PurePosixPath(path).stem


# ── 1. Localization sweeps ───────────────────────────────────

LOC_MAIN = {"dropout": "zjm2ug9y", "noise": "1lsya445"}
LOC_CTRL = {"dropout": "unnbhcwb", "noise": "vay012s7"}
LOC_CTRL_NOISE_EXT = ["l0ugcqlz", "owsfjyp4"]
COL_SENTENCES = "experiment.sentences_file"
COL_PROMPT = "prompt_turns"

loc_main_d = load_sweep(LOC_MAIN["dropout"], PROJECT)
loc_main_n = load_sweep(LOC_MAIN["noise"], PROJECT)
loc_main_d = loc_main_d[loc_main_d[COL_MODEL].isin(MODELS)].copy()
loc_main_n = loc_main_n[loc_main_n[COL_MODEL].isin(MODELS)].copy()

loc_ctrl_d = load_sweep(LOC_CTRL["dropout"], PROJECT)
loc_ctrl_n = load_sweep(LOC_CTRL["noise"], PROJECT)
loc_ctrl_d = loc_ctrl_d[loc_ctrl_d[COL_MODEL].isin(MODELS)].copy()
loc_ctrl_n = loc_ctrl_n[loc_ctrl_n[COL_MODEL].isin(MODELS)].copy()

for sid in LOC_CTRL_NOISE_EXT:
    ext = load_sweep(sid, PROJECT)
    ext = ext[ext[COL_MODEL].isin(MODELS)]
    loc_ctrl_n = pd.concat([loc_ctrl_n, ext], ignore_index=True)

N_PER_RUN = (
    int(loc_main_d["total_samples"].mode().iloc[0])
    if "total_samples" in loc_main_d.columns
    else 1000
)

# Discover sentence lengths and control prompts
sent_files = sorted(loc_main_d[COL_SENTENCES].unique())
sent_map = {_stem(sf): sf for sf in sent_files}
prompt_files = sorted(loc_ctrl_d[COL_PROMPT].unique())
prompt_map = {p.split("/")[-1]: p for p in prompt_files}

LOC_METRICS = [
    m
    for m in [
        "accuracy_primary",
        "accuracy_aggregate",
        "accuracy_argmax",
        "content_accuracy_primary",
        "content_accuracy_aggregate",
        "roc_auc_primary",
        "roc_auc_aggregate",
        "macro_f1_primary",
        "macro_f1_aggregate",
    ]
    if m in loc_main_d.columns
]

LOC_CTRL_METRICS = [m for m in LOC_METRICS if m in loc_ctrl_d.columns]

# ── 2. Zero-shot sweeps ─────────────────────────────────────

CLASS_DROPOUT_SWEEPS = ["rxrpp5xn", "tfhcmgrj", "wt49m58u", "m38olscs", "8yzipesc"]
CLASS_NOISE_SWEEPS = ["ozwrrbx8", "sfn2calp", "e924abmp", "a72eirur", "thhgb8rh"]

zs_all_d = load_and_merge(CLASS_DROPOUT_SWEEPS, PROJECT)
zs_all_n = load_and_merge(CLASS_NOISE_SWEEPS, PROJECT)
zs_all_d = zs_all_d[zs_all_d[COL_MODEL].isin(MODELS)].copy()
zs_all_n = zs_all_n[zs_all_n[COL_MODEL].isin(MODELS)].copy()

N_ZS = 1000

COL_ALIASES = "aliases"
ZS_ALIASES = sorted(zs_all_d[COL_ALIASES].dropna().unique())

ZS_METRICS = [
    m
    for m in [
        "accuracy_primary",
        "accuracy_aggregate",
        "accuracy_argmax",
        "logit_diff_correct",
        "agg_logit_diff_correct",
    ]
    if m in zs_all_d.columns
]

# Zero-shot: preprocess logit_diff columns
_LD_D = "dropout_mean_logit_diff_dropout_vs_noise"
_LD_N = "noise_mean_logit_diff_dropout_vs_noise"
if _LD_D in zs_all_d.columns:
    zs_all_d["logit_diff_correct"] = zs_all_d[_LD_D]
    zs_all_d["logit_diff_correct_se"] = zs_all_d.get(
        "dropout_se_logit_diff_dropout_vs_noise", np.nan
    )
    zs_all_d["logit_diff_correct_sd"] = zs_all_d.get(
        "dropout_std_logit_diff_dropout_vs_noise", np.nan
    )
if _LD_N in zs_all_n.columns:
    zs_all_n["logit_diff_correct"] = -zs_all_n[_LD_N]
    zs_all_n["logit_diff_correct_se"] = zs_all_n.get(
        "noise_se_logit_diff_dropout_vs_noise", np.nan
    )
    zs_all_n["logit_diff_correct_sd"] = zs_all_n.get(
        "noise_std_logit_diff_dropout_vs_noise", np.nan
    )

# ── 3. Few-shot (ICL) sweeps ────────────────────────────────

ICL_MAIN_SWEEPS = ["lwdwtazj", "ri8g7hs9", "ogeqsy4a", "yvcjbhg5"]
COL_SWAP = "prompts.turns.swap_labels"
COL_NUM_PAIRS = "prompts.turns.num_pairs"
COL_ACC = "accuracy_primary"

icl_main = load_and_merge(ICL_MAIN_SWEEPS, PROJECT)
icl_main = icl_main[icl_main[COL_MODEL].isin(MODELS)].copy()

N_ICL = 1000

print(
    f"\nLocalization: {len(loc_main_d)} + {len(loc_main_n)} main runs, "
    f"{len(loc_ctrl_d)} + {len(loc_ctrl_n)} control runs"
)
print(f"  Sentence lengths: {list(sent_map.keys())}")
print(f"  Control prompts: {list(prompt_map.keys())}")
print(f"Zero-shot:   {len(zs_all_d)} + {len(zs_all_n)} runs, aliases: {ZS_ALIASES}")
print(f"ICL:         {len(icl_main)} runs")


Localization: 1200 + 1224 main runs, 1000 + 1130 control runs
  Sentence lengths: ['11tok', '15tok', '19tok', '23tok', '3tok', '7tok']
  Control prompts: ['control_topic_animals_cities', 'control_topic_gardening_vehicles', 'control_topic_ocean_mountain', 'control_topic_sports_music', 'control_topic_weather_technology']
Zero-shot:   803 + 798 runs, aliases: ['beer_wine', 'blanking_fuzzing', 'bralto_sivek', 'cat_dog', 'clipping_smoothing', 'dorvane_kenlo', 'dropout_jitter', 'dropout_noise', 'dropout_quantization', 'fast_slow', 'fire_water', 'foo_bar', 'forest_desert', 'freezing_unfreezing', 'groudon_kyogre', 'hot_cold', 'jedi_sith', 'light_dark', 'linux_windows', 'mario_sonic', 'masking_jitter', 'masking_noise', 'none', 'ocean_mountain', 'pasta_pizza', 'plain_rich', 'playstation_xbox', 'quantization_noise', 'quelp_frando', 'river_lake', 'rotation_permutation', 'scaling_translation', 'sharp_blunt', 'smooth_rough', 'stark_lannister', 'steering_clamping', 'still_sparkling', 'sun_moon', 'su

In [3]:
# ══════════════════════════════════════════════════════════════
# Shared plotting helpers
# ══════════════════════════════════════════════════════════════


def localization_curves(
    df, x_col, metric, n_per_run, models, group_col=None, group_value="all"
):
    """Compute {model: (x, y_pct, se_pct, sd_pct)}.
    group_value == "all" pools across all values of group_col.
    """
    is_pct = "accuracy" in metric or "f1" in metric
    scale = 100 if is_pct else 1
    curves = {}
    for model in models:
        sub = df[df[COL_MODEL] == model]
        if metric not in sub.columns or sub.empty:
            continue
        if group_value != "all" and group_col:
            sub = sub[sub[group_col] == group_value]
        if sub.empty:
            continue
        grp = sub.groupby(x_col)[metric]
        avg = grp.mean().sort_index() * scale
        if avg.empty:
            continue
        if is_pct:
            p = avg / 100
            pq = p * (1 - p)
            n_total = grp.count() * n_per_run
            se = np.sqrt(pq / n_total) * 100
            sd = np.sqrt(pq) * 100
        else:
            se = grp.sem().sort_index() * scale
            sd = grp.std().sort_index() * scale
        curves[model] = (avg.index.values, avg.values, se.values, sd.values)
    return curves


def plot_loc_panel(
    ax,
    curves,
    xlabel,
    models,
    ylabel="Accuracy (%)",
    error_type="SE",
    chance=50,
    ref_lines=None,
    ymin=None,
    ymax=None,
):
    for model in models:
        if model not in curves:
            continue
        x, y, se, sd = curves[model]
        err = se if error_type == "SE" else sd
        c = MODEL_COLORS[model]
        ax.plot(
            x, y, marker="o", markersize=3, lw=2, color=c, label=MODEL_LABELS[model]
        )
        ax.fill_between(x, y - err, y + err, alpha=0.25, color=c)
    if chance is not None:
        ax.axhline(chance, color="gray", ls="--", alpha=0.5, lw=0.8)
    for rl in ref_lines or []:
        ax.axhline(rl, color="gray", ls="--", alpha=0.5, lw=0.8)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
    if ymin is not None and ymax is not None and ymin < ymax:
        ax.set_ylim(ymin, ymax)


def make_model_checkboxes(models=MODELS):
    return {
        m: widgets.Checkbox(value=True, description=MODEL_LABELS[m], indent=False)
        for m in models
    }

---

## 1. Localization — Main Experiment

The model is shown two sentences, one of which has been perturbed (either by
dropout or additive noise). It must pick which sentence was perturbed using
first-token logits (A vs B).

Each curve shows one model's accuracy as a function of perturbation strength,
pooled across all sentence lengths (3 to 23 tokens). Error bands show
$\pm$ SE (Bernoulli).

In [4]:
MAIN_METRICS = [
    ("Accuracy", "accuracy_argmax"),
    ("Accuracy (constrained to A/B)", "accuracy_primary"),
]

w_metric = widgets.Dropdown(
    options=MAIN_METRICS, value="accuracy_argmax", description="Metric:"
)
w_error = widgets.RadioButtons(
    options=["SE", "SD"],
    value="SE",
    description="Band:",
    layout=widgets.Layout(width="auto"),
)
w_models = make_model_checkboxes()
w_ymin = widgets.FloatText(
    value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
)
w_ymax = widgets.FloatText(
    value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
)
out_main = widgets.Output()


def redraw_main(*_):
    out_main.clear_output(wait=True)
    with out_main:
        models = [m for m in MODELS if w_models[m].value]
        metric = w_metric.value
        metric_label = {v: k for k, v in MAIN_METRICS}[metric]
        et = w_error.value
        ymin = w_ymin.value if not np.isnan(w_ymin.value) else None
        ymax = w_ymax.value if not np.isnan(w_ymax.value) else None

        cd = localization_curves(loc_main_d, COL_DROPOUT, metric, N_PER_RUN, models)
        cn = localization_curves(loc_main_n, COL_NOISE, metric, N_PER_RUN, models)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
        plot_loc_panel(
            ax1,
            cd,
            r"Dropout rate $p$",
            models,
            ylabel="Accuracy (%)",
            error_type=et,
            ymin=ymin,
            ymax=ymax,
        )
        plot_loc_panel(
            ax2,
            cn,
            r"Noise SD $\sigma$",
            models,
            ylabel="",
            error_type=et,
            ymin=ymin,
            ymax=ymax,
        )

        handles, labels = ax1.get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc="upper center",
            ncol=max(len(models), 1),
            fontsize=10,
            bbox_to_anchor=(0.5, 1.08),
        )

        fig.suptitle(
            f"Localization — Main ({metric_label}, ±{et})", fontsize=14, y=1.12
        )
        fig.tight_layout()
        plt.show()


for w in [w_metric, w_error, w_ymin, w_ymax]:
    w.observe(redraw_main, names="value")
for cb in w_models.values():
    cb.observe(redraw_main, names="value")

display(
    widgets.HBox([w_metric, w_error]),
    widgets.HBox([w_ymin, w_ymax]),
    widgets.HBox(list(w_models.values())),
    out_main,
)
redraw_main()

/Users/m.bronzi/workspace/llm_mechanistic_detection/.venv/lib/python3.12/site-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


Output()

---

## 2. Localization — Control Experiment

Instead of asking "which sentence was perturbed?", the control uses topic-based
prompts ("which sentence is about animals?"). The model should answer correctly
regardless of perturbation strength — any drop in accuracy would indicate that
the model is distracted by the perturbation rather than answering about the topic.

The dashed line at 95% marks the expected baseline for content-based questions.

In [5]:
cw_error = widgets.RadioButtons(
    options=["SE", "SD"],
    value="SE",
    description="Band:",
    layout=widgets.Layout(width="auto"),
)
cw_models = make_model_checkboxes()
cw_ymin = widgets.FloatText(
    value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
)
cw_ymax = widgets.FloatText(
    value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
)
out_ctrl = widgets.Output()


def redraw_ctrl(*_):
    out_ctrl.clear_output(wait=True)
    with out_ctrl:
        models = [m for m in MODELS if cw_models[m].value]
        metric = "content_accuracy_primary"
        et = cw_error.value
        ymin = cw_ymin.value if not np.isnan(cw_ymin.value) else None
        ymax = cw_ymax.value if not np.isnan(cw_ymax.value) else None

        cd = localization_curves(loc_ctrl_d, COL_DROPOUT, metric, N_PER_RUN, models)
        cn = localization_curves(loc_ctrl_n, COL_NOISE, metric, N_PER_RUN, models)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
        plot_loc_panel(
            ax1,
            cd,
            r"Dropout rate $p$",
            models,
            ylabel="Accuracy (%)",
            error_type=et,
            ref_lines=[95],
            ymin=ymin,
            ymax=ymax,
        )
        plot_loc_panel(
            ax2,
            cn,
            r"Noise SD $\sigma$",
            models,
            ylabel="",
            error_type=et,
            ref_lines=[95],
            ymin=ymin,
            ymax=ymax,
        )

        handles, labels = ax1.get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc="upper center",
            ncol=max(len(models), 1),
            fontsize=10,
            bbox_to_anchor=(0.5, 1.08),
        )

        fig.suptitle(f"Localization \u2014 Control (\u00b1{et})", fontsize=14, y=1.12)
        fig.tight_layout()
        plt.show()


for w in [cw_error, cw_ymin, cw_ymax]:
    w.observe(redraw_ctrl, names="value")
for cb in cw_models.values():
    cb.observe(redraw_ctrl, names="value")

display(
    widgets.HBox([cw_error]),
    widgets.HBox([cw_ymin, cw_ymax]),
    widgets.HBox(list(cw_models.values())),
    out_ctrl,
)
redraw_ctrl()

/Users/m.bronzi/workspace/llm_mechanistic_detection/.venv/lib/python3.12/site-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


Output()

---

## 3. Zero-shot Classification

The model is shown a single sentence (perturbed by either dropout or noise)
and asked to classify the perturbation type. Each sweep applies only one
perturbation type, so all 1000 samples per run are the same type.

Option labels come from different **aliases**:
- **Dropout/Noise** (main): the true perturbation names.
- **Masking/Jitter**: semantically related synonyms.
- **Other controls** (Rotation/Permutation, Vanilla/Chocolate, Foo/Bar, ...):
  unrelated labels that should not convey perturbation identity.

Main and synonym aliases are drawn as solid thick lines; controls as dashed thin
lines. All four models share the same alias color. The x-axis shows perturbation
strength as a percentile (0 = weakest, 100 = strongest) so that dropout and
noise panels are comparable.

In [6]:
# ── Zero-shot: alias styling ─────────────────────────────────

MAIN_STYLE_ALIASES = {"dropout_noise", "masking_jitter"}
MAIN_ALIAS = "dropout_noise"

ALIAS_COLORS = {
    "dropout_noise": "#000000",
    "masking_jitter": "#c0392b",
    "rotation_permutation": "#1a5276",
    "scaling_translation": "#2e86c1",
    "vanilla_chocolate": "#1e7a1e",
    "foo_bar": "#5b2c8c",
    "x_y": "#8e44ad",
    "none": "#777777",
}

ALIAS_DISPLAY = {
    "dropout_noise": "Dropout/Noise",
    "masking_jitter": "Masking/Jitter",
    "rotation_permutation": "Rotation/Permutation",
    "scaling_translation": "Scaling/Translation",
    "vanilla_chocolate": "Vanilla/Chocolate",
    "foo_bar": "Foo/Bar",
    "x_y": "X/Y",
    "none": "None",
}

_extra = plt.cm.tab20.colors
_ci = 0
for a in ZS_ALIASES:
    if a not in ALIAS_COLORS:
        ALIAS_COLORS[a] = "#{:02x}{:02x}{:02x}".format(
            *[int(c * 255) for c in _extra[_ci % 20][:3]]
        )
        _ci += 1


# ── Zero-shot: curve computation ────────────────────────────


def _per_model_pct(x_values):
    n = len(x_values)
    if n <= 1:
        return {x_values[0]: 50.0} if n == 1 else {}
    return {v: i * 100 / (n - 1) for i, v in enumerate(x_values)}


def zs_curves(df, x_col, metric, aliases, n_stochastic, models):
    """Return {alias: {model: (x_pct, y, se, sd)}}."""
    is_pct = "accuracy" in metric or "f1" in metric
    is_ld = "logit_diff" in metric
    scale = 100 if is_pct else 1
    se_col = f"{metric}_se"
    sd_col = f"{metric}_sd"
    result = {}
    for alias in aliases:
        sub_alias = df[df[COL_ALIASES] == alias]
        curves = {}
        for model in models:
            sub = sub_alias[sub_alias[COL_MODEL] == model]
            if metric not in sub.columns or sub.empty:
                continue
            grp = sub.groupby(x_col)
            avg = grp[metric].mean().sort_index() * scale
            if avg.empty:
                continue
            if is_pct:
                p = avg / 100
                pq = p * (1 - p)
                n_total = grp[metric].count() * n_stochastic
                se = np.sqrt(pq / n_total) * 100
                sd = np.sqrt(pq) * 100
            elif is_ld and se_col in sub.columns:
                n_runs = grp[metric].count()
                if (n_runs > 1).any():
                    se = grp[metric].sem().sort_index()
                    sd = grp[sd_col].mean().sort_index()
                else:
                    se = grp[se_col].first().sort_index()
                    sd = grp[sd_col].first().sort_index()
            else:
                se = grp[metric].sem().sort_index() * scale
                sd = grp[metric].std().sort_index() * scale
            x_raw = avg.index.values
            pct_map = _per_model_pct(x_raw)
            x_pct = np.array([pct_map[v] for v in x_raw])
            curves[model] = (x_pct, avg.values, se.values, sd.values)
        if curves:
            result[alias] = curves
    return result


def plot_zs_panel(
    ax,
    all_curves,
    xlabel,
    models,
    error_type="SE",
    ylabel="Accuracy (%)",
    ymin=None,
    ymax=None,
):
    is_pct = "accuracy" in ylabel.lower()
    for alias, curves in all_curves.items():
        main_style = alias in MAIN_STYLE_ALIASES
        c = ALIAS_COLORS.get(alias, "#888888")
        ls = "-" if main_style else "--"
        lw = 2.5 if main_style else 1.2
        ms = 4 if main_style else 0
        alpha_band = 0.25 if main_style else 0.12
        zorder = 10 if main_style else 1
        for model in models:
            if model not in curves:
                continue
            x, y, se, sd = curves[model]
            err = se if error_type == "SE" else sd
            ax.plot(
                x, y, marker="o", markersize=ms, lw=lw, color=c, ls=ls, zorder=zorder
            )
            ax.fill_between(
                x, y - err, y + err, alpha=alpha_band, color=c, zorder=zorder - 1
            )
    if is_pct:
        ax.axhline(50, color="gray", ls=":", alpha=0.5, lw=0.8)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xlim(-2, 102)
    ax.set_xticks(range(0, 101, 10))
    ax.grid(True, alpha=0.25)
    if ymin is not None and ymax is not None and ymin < ymax:
        ax.set_ylim(ymin, ymax)

In [7]:
ZS_METRIC_OPTS = [
    ("Accuracy", "accuracy_argmax"),
    ("Accuracy (constrained to A/B)", "accuracy_primary"),
]

ZS_MODEL_OPTS = [(MODEL_LABELS[m], m) for m in MODELS]

zw_metric = widgets.Dropdown(
    options=ZS_METRIC_OPTS, value="accuracy_argmax", description="Metric:"
)
zw_error = widgets.RadioButtons(
    options=["SE", "SD"],
    value="SE",
    description="Band:",
    layout=widgets.Layout(width="auto"),
)
zw_model = widgets.RadioButtons(
    options=ZS_MODEL_OPTS,
    value=MODELS[0],
    description="Model:",
    layout=widgets.Layout(width="auto"),
)
zw_ymin = widgets.FloatText(
    value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
)
zw_ymax = widgets.FloatText(
    value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
)

# ── Alias checkboxes: split into shared (all models) and Qwen 3 32B only ──


def _alias_label(a):
    return ALIAS_DISPLAY.get(a, a)


# Compute which aliases are available per model
_alias_per_model = {
    m: set(
        list(zs_all_d[zs_all_d[COL_MODEL] == m][COL_ALIASES].dropna().unique())
        + list(zs_all_n[zs_all_n[COL_MODEL] == m][COL_ALIASES].dropna().unique())
    )
    for m in MODELS
}
_shared_aliases = sorted(set.intersection(*_alias_per_model.values()))
_qwen32_only = sorted(_alias_per_model["qwen3_32b"] - set(_shared_aliases))

zw_alias = {}
for a in _shared_aliases + _qwen32_only:
    zw_alias[a] = widgets.Checkbox(
        value=(a == MAIN_ALIAS),
        description=_alias_label(a),
        indent=False,
        layout=widgets.Layout(width="180px"),
    )

zw_btn_all = widgets.Button(
    description="Select all", layout=widgets.Layout(width="100px")
)
zw_btn_none = widgets.Button(
    description="Clear all", layout=widgets.Layout(width="100px")
)
zw_btn_main = widgets.Button(
    description="Main only", layout=widgets.Layout(width="100px")
)


def _on_all(_):
    for cb in zw_alias.values():
        cb.value = True


def _on_none(_):
    for cb in zw_alias.values():
        cb.value = False


def _on_main(_):
    for cb in zw_alias.values():
        cb.value = False
    zw_alias[MAIN_ALIAS].value = True


zw_btn_all.on_click(_on_all)
zw_btn_none.on_click(_on_none)
zw_btn_main.on_click(_on_main)

# Build the two labeled sections
_shared_grid = widgets.GridBox(
    [zw_alias[a] for a in _shared_aliases],
    layout=widgets.Layout(grid_template_columns="repeat(5, 180px)"),
)
_qwen32_grid = widgets.GridBox(
    [zw_alias[a] for a in _qwen32_only],
    layout=widgets.Layout(grid_template_columns="repeat(5, 180px)"),
)

_alias_panel = widgets.VBox(
    [
        widgets.HBox([zw_btn_all, zw_btn_none, zw_btn_main]),
        widgets.HTML("<b>All models:</b>"),
        _shared_grid,
        widgets.HTML("<b>Qwen 3 32B only:</b>"),
        _qwen32_grid,
    ]
)

out_zs = widgets.Output()


def redraw_zs(*_):
    out_zs.clear_output(wait=True)
    with out_zs:
        models = [zw_model.value]
        sel = [a for a in _shared_aliases + _qwen32_only if zw_alias[a].value]
        metric = zw_metric.value
        metric_label = {v: k for k, v in ZS_METRIC_OPTS}[metric]
        et = zw_error.value
        ymin = zw_ymin.value if not np.isnan(zw_ymin.value) else None
        ymax = zw_ymax.value if not np.isnan(zw_ymax.value) else None

        if not sel:
            print("No aliases selected.")
            return

        cd = zs_curves(zs_all_d, COL_DROPOUT, metric, sel, N_ZS, models)
        cn = zs_curves(zs_all_n, COL_NOISE, metric, sel, N_ZS, models)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
        plot_zs_panel(
            ax1,
            cd,
            "Perturbation percentile (dropout)",
            models,
            error_type=et,
            ylabel="Accuracy (%)",
            ymin=ymin,
            ymax=ymax,
        )
        plot_zs_panel(
            ax2,
            cn,
            "Perturbation percentile (noise)",
            models,
            error_type=et,
            ylabel="",
            ymin=ymin,
            ymax=ymax,
        )

        alias_handles = []
        for a in sel:
            if a in cd or a in cn:
                main = a in MAIN_STYLE_ALIASES
                alias_handles.append(
                    Line2D(
                        [0],
                        [0],
                        color=ALIAS_COLORS.get(a, "#888"),
                        lw=2.5 if main else 1.2,
                        ls="-" if main else "--",
                        label=_alias_label(a),
                    )
                )
        fig.legend(
            handles=alias_handles,
            loc="upper center",
            ncol=min(len(sel), 6),
            fontsize=9,
            bbox_to_anchor=(0.5, 1.08),
        )

        n_sel = len(sel)
        alias_summary = (
            "main only"
            if sel == [MAIN_ALIAS]
            else f"{n_sel} aliases"
            if n_sel > 2
            else " + ".join(_alias_label(a) for a in sel)
        )
        model_label = MODEL_LABELS[zw_model.value]
        fig.suptitle(
            f"Zero-shot Classification \u2014 {model_label} ({metric_label}, {alias_summary}, \u00b1{et})",
            fontsize=14,
            y=1.12,
        )
        fig.tight_layout()
        plt.show()


for w in [zw_metric, zw_error, zw_model, zw_ymin, zw_ymax]:
    w.observe(redraw_zs, names="value")
for cb in zw_alias.values():
    cb.observe(redraw_zs, names="value")

display(
    widgets.HBox([zw_metric, zw_error, zw_model]),
    widgets.HBox([zw_ymin, zw_ymax]),
    _alias_panel,
    out_zs,
)
redraw_zs()

/Users/m.bronzi/workspace/llm_mechanistic_detection/.venv/lib/python3.12/site-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


Output()

---

## 4. ICL Learning Dynamics

Each ICL run has 1000 stochastic samples (500 dropout + 500 noise). The sweep
grid varies dropout rate (11 values), noise std (11 values), num_pairs
(1, 3, 5, 7, 9 teaching examples), and swap_labels (True/False).

For each (model, num_pairs), accuracy is averaged over the full 11 x 11 grid
of perturbation strengths (121 runs, 121,000 trials).

- **Solid lines**: swap=False (standard label assignment).
- **Dashed lines**: swap=True (flipped labels, accuracy reflected as 100 - acc
  so that high = good under the original scheme).

Error bands show $\pm$ SE (Bernoulli, pooled over the 121,000 trials).

In [8]:
# ── ICL: compute learning tables ─────────────────────────────

# Bernoulli SE and SD for accuracy
for df in [icl_main]:
    p = df[COL_ACC]
    pq = p * (1 - p)
    df["accuracy_se"] = np.sqrt(pq / N_ICL)
    df["accuracy_sd"] = np.sqrt(pq)

# Logit diff columns (if present)
_LD_D_ICL = "dropout_mean_logit_diff_dropout"
_LD_N_ICL = "noise_mean_logit_diff_noise"
has_ld = _LD_D_ICL in icl_main.columns and _LD_N_ICL in icl_main.columns
if has_ld:
    icl_main["logit_diff_correct"] = (icl_main[_LD_D_ICL] + icl_main[_LD_N_ICL]) / 2
    icl_main["logit_diff_correct_sd"] = np.sqrt(
        (
            icl_main.get("dropout_std_logit_diff_dropout", 0) ** 2
            + icl_main.get("noise_std_logit_diff_noise", 0) ** 2
        )
        / 2
    )

N_GRID = 121  # 11 dropout x 11 noise


def compute_learning_table(df, swap_value):
    sub = df[df[COL_SWAP] == swap_value]
    rows = []
    for model in MODELS:
        ms = sub[sub[COL_MODEL] == model]
        for np_val in sorted(ms[COL_NUM_PAIRS].dropna().unique()):
            chunk = ms[ms[COL_NUM_PAIRS] == np_val]
            acc = chunk[COL_ACC]
            n_runs = len(acc)
            if n_runs == 0:
                continue
            p_mean = acc.mean()
            pq = p_mean * (1 - p_mean)
            n_total = n_runs * N_ICL

            row = {
                "model": model,
                "num_pairs": int(np_val),
                "n_runs": n_runs,
                "acc_pct": p_mean * 100,
                "acc_pooled_se_pct": np.sqrt(pq / n_total) * 100,
                "acc_sd_pct": np.sqrt(pq) * 100,
                "acc_empirical_se_pct": acc.std() / np.sqrt(n_runs) * 100,
            }

            if has_ld:
                ld = chunk["logit_diff_correct"]
                ld_sd = chunk["logit_diff_correct_sd"]
                row.update(
                    {
                        "ld_mean": ld.mean(),
                        "ld_pooled_se": ld_sd.mean() / np.sqrt(n_total),
                        "ld_sd": ld_sd.mean(),
                        "ld_empirical_se": ld.std() / np.sqrt(n_runs),
                    }
                )
            rows.append(row)
    return pd.DataFrame(rows)


ldt_false = compute_learning_table(icl_main, swap_value=False)
ldt_true = compute_learning_table(icl_main, swap_value=True)

# Flip swap=True: high = good under original labels
ldt_true["acc_pct_flipped"] = 100 - ldt_true["acc_pct"]
ldt_true["acc_pooled_se_pct_flipped"] = ldt_true["acc_pooled_se_pct"]
ldt_true["acc_empirical_se_pct_flipped"] = ldt_true["acc_empirical_se_pct"]
ldt_true["acc_sd_pct_flipped"] = ldt_true["acc_sd_pct"]
if has_ld:
    ldt_true["ld_mean_flipped"] = -ldt_true["ld_mean"]
    ldt_true["ld_pooled_se_flipped"] = ldt_true["ld_pooled_se"]
    ldt_true["ld_empirical_se_flipped"] = ldt_true["ld_empirical_se"]
    ldt_true["ld_sd_flipped"] = ldt_true["ld_sd"]

# ── ICL: interactive plot ────────────────────────────────────

ld_w_models = make_model_checkboxes()
ld_w_swap = widgets.RadioButtons(
    options=["swap=False only", "swap=True only", "Both"],
    value="swap=False only",
    description="Swap:",
    layout=widgets.Layout(width="auto"),
)
ld_w_ymin = widgets.FloatText(
    value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
)
ld_w_ymax = widgets.FloatText(
    value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
)
ld_out = widgets.Output()


def _ld_cols(met, et, flipped=False):
    sfx = "_flipped" if flipped else ""
    if met == "Accuracy":
        y_col = f"acc_pct{sfx}"
        err_col = (
            f"acc_pooled_se_pct{sfx}"
            if et == "Pooled SE"
            else f"acc_empirical_se_pct{sfx}"
            if et == "Empirical SE"
            else f"acc_sd_pct{sfx}"
        )
    else:
        y_col = f"ld_mean{sfx}"
        err_col = (
            f"ld_pooled_se{sfx}"
            if et == "Pooled SE"
            else f"ld_empirical_se{sfx}"
            if et == "Empirical SE"
            else f"ld_sd{sfx}"
        )
    return y_col, err_col


def redraw_ld(*_):
    ld_out.clear_output(wait=True)
    with ld_out:
        models = [m for m in MODELS if ld_w_models[m].value]
        met = "Accuracy"
        et = "Pooled SE"
        swap_mode = ld_w_swap.value
        ymin = ld_w_ymin.value if not np.isnan(ld_w_ymin.value) else None
        ymax = ld_w_ymax.value if not np.isnan(ld_w_ymax.value) else None

        is_pct = met == "Accuracy"
        ylabel = "Accuracy (%)" if is_pct else "Logit diff (correct \u2212 incorrect)"
        show_false = swap_mode in ("swap=False only", "Both")
        show_true = swap_mode in ("swap=True only", "Both")

        fig, ax = plt.subplots(figsize=(10, 6))

        if show_false:
            y_col, err_col = _ld_cols(met, et, flipped=False)
            if y_col in ldt_false.columns:
                for model in models:
                    sub = ldt_false[ldt_false["model"] == model].sort_values(
                        "num_pairs"
                    )
                    if sub.empty:
                        continue
                    c = MODEL_COLORS[model]
                    ax.plot(
                        sub["num_pairs"],
                        sub[y_col],
                        marker="o",
                        markersize=5,
                        lw=2.5,
                        color=c,
                        zorder=10,
                    )
                    ax.fill_between(
                        sub["num_pairs"],
                        sub[y_col] - sub[err_col],
                        sub[y_col] + sub[err_col],
                        alpha=0.25,
                        color=c,
                        zorder=9,
                    )

        if show_true:
            y_col, err_col = _ld_cols(met, et, flipped=True)
            if y_col in ldt_true.columns:
                for model in models:
                    sub = ldt_true[ldt_true["model"] == model].sort_values("num_pairs")
                    if sub.empty:
                        continue
                    c = MODEL_COLORS[model]
                    ax.plot(
                        sub["num_pairs"],
                        sub[y_col],
                        marker="o",
                        markersize=3,
                        lw=1.5,
                        color=c,
                        ls="--",
                        alpha=0.7,
                        zorder=5,
                    )
                    ax.fill_between(
                        sub["num_pairs"],
                        sub[y_col] - sub[err_col],
                        sub[y_col] + sub[err_col],
                        alpha=0.12,
                        color=c,
                        zorder=4,
                    )

        handles = [
            Line2D(
                [0],
                [0],
                color=MODEL_COLORS[m],
                lw=2.5,
                marker="o",
                markersize=4,
                label=MODEL_LABELS[m],
            )
            for m in models
        ]
        if show_false and show_true:
            handles.append(
                Line2D([0], [0], color="gray", lw=2.5, ls="-", label="swap=False")
            )
            handles.append(
                Line2D(
                    [0], [0], color="gray", lw=1.5, ls="--", label="swap=True (flipped)"
                )
            )

        ref = 50 if is_pct else 0
        ax.axhline(ref, color="gray", ls=":", alpha=0.5, lw=0.8)
        ax.set_xlabel("Number of teaching examples")
        ax.set_ylabel(ylabel)
        ax.set_xticks(sorted(ldt_false["num_pairs"].unique()))
        ax.grid(True, alpha=0.25)
        ax.legend(handles=handles, fontsize=9, framealpha=0.9)
        if ymin is not None and ymax is not None and ymin < ymax:
            ax.set_ylim(ymin, ymax)

        fig.suptitle("ICL Learning Dynamics (\u00b1 Pooled SE)", fontsize=14)
        fig.tight_layout()
        plt.show()


for w in [ld_w_swap, ld_w_ymin, ld_w_ymax]:
    w.observe(redraw_ld, names="value")
for cb in ld_w_models.values():
    cb.observe(redraw_ld, names="value")

display(
    widgets.HBox([ld_w_swap, ld_w_ymin, ld_w_ymax]),
    widgets.HBox(list(ld_w_models.values())),
    ld_out,
)
redraw_ld()

/Users/m.bronzi/workspace/llm_mechanistic_detection/.venv/lib/python3.12/site-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


Output()